In [3]:
from ppktstore.registry import configure_phenopacket_registry
import pandas as pd
from phenosign import PhenotypeDatasetBuilder, HPOCorrelationAnalyzer

## Loading cohorts from phenopacket store

In [4]:
def run_global_cohort_scan():

    registry = configure_phenopacket_registry()

    significant_cohorts_summary = []
    all_sig_tables = []

    with registry.open_phenopacket_store() as ps:
        available_cohorts = [c.name for c in ps.cohorts()]
        print(f"Total cohorts found: {len(available_cohorts)}")

        for i, cohort_name in enumerate(available_cohorts, 1):
            print(f"[{i}/{len(available_cohorts)}] Processing cohort: {cohort_name}")

            try:
                phenopackets = list(ps.iter_cohort_phenopackets(cohort_name))

                if len(phenopackets) < 20:
                    continue

                dataset = PhenotypeDatasetBuilder(phenopackets).build(
                    build_gpsea_cohort=False
                )

                analyzer = HPOCorrelationAnalyzer(dataset=dataset)
                results_table = analyzer.compute_correlation_matrix().results_table

                if results_table.empty:
                    continue

                sig_results = results_table[results_table["adj_p_value"] < 0.05]
                if len(sig_results)==0:
                    continue

                summary_row = {
                    "cohort": cohort_name,
                    "n_patients": len(phenopackets),
                    "tested_pairs": len(results_table),
                    "adj_signif_counts": len(sig_results),
                }

                significant_cohorts_summary.append(summary_row)

                if len(sig_results):
                    all_sig_tables.append(sig_results.assign(Cohort=cohort_name))

            except Exception:
                continue

    summary_df = pd.DataFrame(significant_cohorts_summary)

    master_sig_df = (
        pd.concat(all_sig_tables, ignore_index=True)
        if all_sig_tables else pd.DataFrame()
    )

    return summary_df, master_sig_df

In [5]:
summary_df, master_sig_df = run_global_cohort_scan()

Total cohorts found: 700
[1/700] Processing cohort: 11q_terminal_deletion


Calculating pairwise correlation: 100%|██████████| 2016/2016 [00:01<00:00, 1386.80it/s]


[2/700] Processing cohort: AAGAB
[3/700] Processing cohort: ABCA4
[4/700] Processing cohort: ABCB7
[5/700] Processing cohort: ABCC8
[6/700] Processing cohort: ACBD6


Calculating pairwise correlation: 100%|██████████| 2823/2823 [00:00<00:00, 11433.73it/s]


[7/700] Processing cohort: ACP4
[8/700] Processing cohort: ACTA1
[9/700] Processing cohort: ACTB
[10/700] Processing cohort: ACTN1
[11/700] Processing cohort: ACVR1
[12/700] Processing cohort: ADA
[13/700] Processing cohort: ADA2
[14/700] Processing cohort: ADAMTS10
[15/700] Processing cohort: ADAMTS15
[16/700] Processing cohort: ADAMTSL2
[17/700] Processing cohort: ADGRG1
[18/700] Processing cohort: ADRA2A
[19/700] Processing cohort: AEBP1
[20/700] Processing cohort: AFG3L2


Calculating pairwise correlation: 100%|██████████| 158/158 [00:00<00:00, 4856.05it/s]


... (109 outputs left out)

Calculating pairwise correlation: 100%|██████████| 7/7 [00:00<00:00, 15025.65it/s]


[420/700] Processing cohort: NPC1
[421/700] Processing cohort: NPHS1
[422/700] Processing cohort: NPR2
[423/700] Processing cohort: NPSR1
[424/700] Processing cohort: NR6A1
[425/700] Processing cohort: NRAS
[426/700] Processing cohort: NRL
[427/700] Processing cohort: NRXN1


Calculating pairwise correlation: 100%|██████████| 1000/1000 [00:00<00:00, 7994.45it/s]


[428/700] Processing cohort: NSUN2
[429/700] Processing cohort: NSUN6
[430/700] Processing cohort: NT5C2
[431/700] Processing cohort: NUP210L
[432/700] Processing cohort: NUP54
[433/700] Processing cohort: OAS2
[434/700] Processing cohort: OCA2
[435/700] Processing cohort: OFD1
[436/700] Processing cohort: OPA1
[437/700] Processing cohort: OSBPL2
[438/700] Processing cohort: OTUD6B
[439/700] Processing cohort: OTUD7A
[440/700] Processing cohort: OTULIN
[441/700] Processing cohort: PABPC1L
[442/700] Processing cohort: PACSIN3
[443/700] Processing cohort: PAN2
[444/700] Processing cohort: PANK2
[445/700] Processing cohort: PARK7
[446/700] Processing cohort: PAX3
[447/700] Processing cohort: PAX4
[448/700] Processing cohort: PCDH19
[449/700] Processing cohort: PCYT1A
[450/700] Processing cohort: PDCD1
[451/700] Processing cohort: PDGFRB
[452/700] Processing cohort: PGBD5
[453/700] Processing cohort: PI4K2A
[454/700] Processing cohort: PIEZO2
[455/700] Processing cohort: PIGA
[456/700] Pro

In [6]:
display(summary_df.sort_values("adj_signif_counts", ascending=False))

summary_df.to_csv("cohort_summary.csv", index=False)

if not master_sig_df.empty:
    display(master_sig_df.head(20))
    master_sig_df.to_csv("all_significant_pairs.csv", index=False)

,cohort,n_patients,tested_pairs,adj_signif_counts
18,LMNA,259,1152,205
8,FBN1,143,477,52
17,ITPR1,170,427,38
23,NF1,405,412,27
11,GLI3,81,82,20
9,FBXL4,95,1117,20
16,INPP4A,30,102,17
21,NBAS,67,409,14
7,EPG5,39,38,11
2,ANKRD11,333,895,9


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids,Cohort
0,HP:0000508,Ptosis,HP:0008551,Microtia,0.535656,9.865830e-06,0.008726,19,4,12,34,69,1,15266616,11q_terminal_deletion
1,HP:0000322,Short philtrum,HP:0000343,Long philtrum,-0.547376,6.490005e-06,0.008726,16,36,16,1,69,1,15266616,11q_terminal_deletion
2,HP:0000445,Wide nose,HP:0005280,Depressed nasal bridge,1.000000,7.989277e-10,0.000002,12,0,0,24,36,1,37951597,ACBD6
3,HP:0000280,Coarse facial features,HP:0000729,Autistic behavior,0.846154,3.164882e-05,0.025346,11,0,2,11,24,1,37951597,ACBD6
4,HP:0000280,Coarse facial features,HP:0001257,Spasticity,0.686607,4.023195e-05,0.025346,9,6,0,22,37,1,37951597,ACBD6
5,HP:0000280,Coarse facial features,HP:0002061,Lower limb spasticity,0.717137,3.816877e-05,0.025346,9,5,0,20,34,1,37951597,ACBD6
6,HP:0000403,Recurrent otitis media,HP:0000405,Conductive hearing impairment,0.755012,6.694051e-07,0.000206,27,2,3,13,45,1,36446582,ANKRD11
7,HP:0000405,Conductive hearing impairment,HP:0000407,Sensorineural hearing impairment,1.000000,4.709085e-07,0.000206,82,0,0,4,86,1,36446582,ANKRD11
8,HP:0001642,Pulmonic stenosis,HP:0001647,Bicuspid aortic valve,0.794048,6.903526e-07,0.000206,167,1,1,4,173,1,36446582,ANKRD11
9,HP:0000316,Hypertelorism,HP:0000411,Protruding ear,0.603683,6.199220e-06,0.001387,21,10,2,23,56,1,36446582,ANKRD11
